In [0]:
import requests
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, TimestampType
from pyspark.sql.functions import current_timestamp, lit

file_path = "/Volumes/workspace/stock_data/api_keys/ms_api_key.txt"

with open(file_path, "r") as f:
    api_key = f.read().strip()

tickers = "AAPL,TSLA,AMZN,MSFT,NVDA,GOOGL,META,NFLX,JPM,V,BAC,AMD,PYPL,DIS,T,PFE,COST,INTC,KO,TGT,NKE,SPY,BA,BABA,XOM,WMT,GE,CSCO,VZ,JNJ,CVX,PLTR,SQ,SHOP,SBUX,SOFI,HOOD,RBLX,SNAP,UBER,FDX,ABBV,ETSY,MRNA,LMT,GM,F,RIVN,LCID,CCL,DAL,UAL,AAL,TSM,SONY,ET,NOK,MRO,COIN,SIRI,RIOT,CPRX,VWO,SPYG,ROKU,VIAC,ATVI,BIDU,DOCU,ZM,PINS,TLRY,WBA,MGM,NIO,C,GS,WFC,ADBE,PEP,UNH,CARR,FUBO,HCA,TWTR,BILI,RKT"


def get_stock_data(tickers, api_key):
    url = "https://api.marketstack.com/v2/eod"
    limit = 1000
    offset = 0
    raw_rows = []

    while True:
        params = {
            'symbols': f'{tickers}',
            'access_key': f'{api_key}',
            'date_from' : '2016-01-01',
            'date_to' : '2025-12-15',
            'limit' : limit,
            'offset' : offset,
            "sort": "ASC"
        }

        response = requests.get(url, params=params)
        response.raise_for_status()
        payload = response.json()

        for record in payload.get('data', []):
            raw_rows.append(Row(**{k: str(v) for k, v in record.items()}))

        if offset + limit >= payload["pagination"]["total"]:
            break

        offset += limit
    
    return raw_rows

raw_data = get_stock_data(tickers, api_key)

bronze_schema = StructType([
    StructField("open", StringType(), True),
    StructField("high", StringType(), True),
    StructField("low", StringType(), True),
    StructField("close", StringType(), True),
    StructField("volume", StringType(), True),
    StructField("adj_high", StringType(), True),
    StructField("adj_low", StringType(), True),
    StructField("adj_close", StringType(), True),
    StructField("adj_open", StringType(), True),
    StructField("adj_volume", StringType(), True),
    StructField("split_factor", StringType(), True),
    StructField("dividend", StringType(), True),
    StructField("name", StringType(), True),
    StructField("exchange_code", StringType(), True),
    StructField("asset_type", StringType(), True),
    StructField("price_currency", StringType(), True),
    StructField("symbol", StringType(), True),
    StructField("exchange", StringType(), True),
    StructField("date", StringType(), True)
])

bronze_df = spark.createDataFrame(raw_data, schema=bronze_schema)

bronze_df = (
    bronze_df
    .withColumn("ingest_timestamp", current_timestamp())
    .withColumn("data_source", lit("marketstack_api"))
)

bronze_df.write.format("delta").mode("overwrite").saveAsTable('workspace.stock_data.stock_data_bronze')

In [0]:
%sql
SELECT *
FROM workspace.stock_data.stock_data_bronze
LIMIT 25
